In [73]:
import pandas as pd

In [74]:
stats = pd.read_csv("player_mvp_stats.csv")

In [75]:
stats

,Unnamed: 0,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,...,Pts Max,Share,Team,W,L,W/L%,GB,PS/G,PA/G,SRS
0,0,A.C. Green,PF,27,LAL,82,21,26.4,3.1,6.6,...,0.0,0.0,Los Angeles Lakers,58,24,0.707,5.0,106.3,99.6,6.73
1,1,Byron Scott,SG,29,LAL,82,82,32.1,6.1,12.8,...,0.0,0.0,Los Angeles Lakers,58,24,0.707,5.0,106.3,99.6,6.73
2,2,Elden Campbell,PF,22,LAL,52,0,7.3,1.1,2.4,...,0.0,0.0,Los Angeles Lakers,58,24,0.707,5.0,106.3,99.6,6.73
3,3,Irving Thomas,PF,25,LAL,26,0,4.2,0.7,1.9,...,0.0,0.0,Los Angeles Lakers,58,24,0.707,5.0,106.3,99.6,6.73
4,4,James Worthy,SF,29,LAL,78,74,38.6,9.2,18.7,...,0.0,0.0,Los Angeles Lakers,58,24,0.707,5.0,106.3,99.6,6.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14087,14087,Spencer Hawes,PF,28,MIL,54,1,14.8,2.5,5.1,...,0.0,0.0,Milwaukee Bucks,42,40,0.512,9.0,103.6,103.8,-0.45
14088,14088,Steve Novak,PF,33,MIL,8,0,2.8,0.3,0.9,...,0.0,0.0,Milwaukee Bucks,42,40,0.512,9.0,103.6,103.8,-0.45
14089,14089,Terrence Jones,PF,25,MIL,54,12,23.5,4.3,9.1,...,0.0,0.0,Milwaukee Bucks,42,40,0.512,9.0,103.6,103.8,-0.45
14090,14090,Thon Maker,C,19,MIL,57,34,9.9,1.5,3.2,...,0.0,0.0,Milwaukee Bucks,42,40,0.512,9.0,103.6,103.8,-0.45


In [76]:
del stats["Unnamed: 0"]

In [77]:
pd.isnull(stats).sum()

Player        0
Pos           0
Age           0
Tm            0
G             0
GS            0
MP            0
FG            0
FGA           0
FG%          50
3P            0
3PA           0
3P%        2042
2P            0
2PA           0
2P%          84
eFG%         50
FT            0
FTA           0
FT%         462
ORB           0
DRB           0
TRB           0
AST           0
STL           0
BLK           0
TOV           0
PF            0
PTS           0
Year          0
Pts Won       0
Pts Max       0
Share         0
Team          0
W             0
L             0
W/L%          0
GB            0
PS/G          0
PA/G          0
SRS           0
dtype: int64

In [78]:
stats[pd.isnull(stats["3P%"])][["Player", "3PA"]]

,Player,3PA
2,Elden Campbell,0.0
3,Irving Thomas,0.0
18,Jack Haley,0.0
20,Keith Owens,0.0
30,Benoit Benjamin,0.0
...,...,...
14061,Evan Eschmeyer,0.0
14062,Gheorghe Mureșan,0.0
14064,Jim McIlvaine,0.0
14070,Mark Hendrickson,0.0


In [79]:
stats = stats.fillna(0)

In [80]:
stats.columns

Index(['Player', 'Pos', 'Age', 'Tm', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P',
       '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB',
       'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Year',
       'Pts Won', 'Pts Max', 'Share', 'Team', 'W', 'L', 'W/L%', 'GB', 'PS/G',
       'PA/G', 'SRS'],
      dtype='object')

In [81]:
# Numeric columns are the ones we want to use as predictors. Don't use pts won/pts max/share - these are very related with what we are trying to predict (overfitting)
predictors = ['Age', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P',
       '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB',
       'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Year',
       'W', 'L', 'W/L%', 'GB', 'PS/G',
       'PA/G', 'SRS']

In [108]:
train = stats[stats["Year"] < 2021]

In [109]:
# Don't test on data that is before data you're training on. That's information the algo wouldn't have had in the real world. This causes overfitting.
test = stats[stats["Year"] == 2021]

In [110]:
# Ridge regression shrinks linear regression coefficients to avoid overfitting
from sklearn.linear_model import Ridge
# Alpha is how much the linear regression coefficient is shrunk to prevent overfitting
reg = Ridge(alpha = .1)

In [111]:
# Use these predictors to try to predict share
# SHOULD SCALE FEATURES BEFORE RUNNING RIDGE REGRESSION
reg.fit(train[predictors], train["Share"])

Ridge(alpha=0.1)

In [112]:
reg.fit(train[predictors], train["Share"])
predictions = reg.predict(test[predictors])

In [87]:
predictions = pd.DataFrame(predictions, columns = ['predictions'], index = test.index)

In [88]:
predictions

,predictions
630,0.013567
631,-0.013756
632,0.002414
633,-0.004421
634,0.010734
...,...
13897,-0.012571
13898,-0.011575
13899,0.016424
13900,-0.020434


In [89]:
combination = pd.concat([test[["Player", "Share"]], predictions], axis = 1)

In [90]:
combination

,Player,Share,predictions
630,Aaron Gordon,0.0,0.013567
631,Austin Rivers,0.0,-0.013756
632,Bol Bol,0.0,0.002414
633,Facundo Campazzo,0.0,-0.004421
634,Greg Whittington,0.0,0.010734
...,...,...,...
13897,Patty Mills,0.0,-0.012571
13898,Quinndary Weatherspoon,0.0,-0.011575
13899,Rudy Gay,0.0,0.016424
13900,Tre Jones,0.0,-0.020434


In [91]:
# Jokic won MVP but did not have the highest predicted amount - 2021 is the year before this year (last year we had full data for)
combination.sort_values("Share", ascending = False).head(10)

,Player,Share,predictions
641,Nikola Jokić,0.961,0.154307
8624,Joel Embiid,0.580,0.162713
3651,Stephen Curry,0.449,0.142386
9907,Giannis Antetokounmpo,0.345,0.207436
1389,Chris Paul,0.138,0.072294
10997,Luka Dončić,0.042,0.151430
7464,Damian Lillard,0.038,0.116303
3536,Julius Randle,0.020,0.088878
3531,Derrick Rose,0.010,0.033000
11358,Rudy Gobert,0.008,0.095349


In [92]:
from sklearn.metrics import mean_squared_error

# not meaningful since many of these values are 0
mean_squared_error(combination["Share"], combination["predictions"])

0.0026668954567099475

In [93]:
combination = combination.sort_values("Share", ascending=False)
# adding a rank for each entry in the dataframe
combination["Rk"] = list(range(1, combination.shape[0]+1))

In [94]:
combination.head(10)

,Player,Share,predictions,Rk
641,Nikola Jokić,0.961,0.154307,1
8624,Joel Embiid,0.580,0.162713,2
3651,Stephen Curry,0.449,0.142386,3
9907,Giannis Antetokounmpo,0.345,0.207436,4
1389,Chris Paul,0.138,0.072294,5
10997,Luka Dončić,0.042,0.151430,6
7464,Damian Lillard,0.038,0.116303,7
3536,Julius Randle,0.020,0.088878,8
3531,Derrick Rose,0.010,0.033000,9
11358,Rudy Gobert,0.008,0.095349,10


In [95]:
combination = combination.sort_values("predictions", ascending = False)
combination["Predicted_Rk"] = list(range(1, combination.shape[0] + 1))

In [96]:
combination.head(10)

,Player,Share,predictions,Rk,Predicted_Rk
9907,Giannis Antetokounmpo,0.345,0.207436,4,1
8624,Joel Embiid,0.580,0.162713,2,2
641,Nikola Jokić,0.961,0.154307,1,3
10997,Luka Dončić,0.042,0.151430,6,4
3736,LeBron James,0.001,0.147511,15,5
3651,Stephen Curry,0.449,0.142386,3,6
4177,Kevin Durant,0.000,0.141350,531,7
4174,James Harden,0.001,0.140598,13,8
11784,Zion Williamson,0.000,0.127926,251,9
3876,Russell Westbrook,0.005,0.120228,11,10


In [97]:
# Error metric - we really care about who is in the top 5; of all the final vote getters in the top 5, how many did we actually place in the top 5?
# We will use average precision - not commonly used for regression, because it deals w/ ranking, but this problem requires ranking

combination.sort_values("Share", ascending=False).head(10)


,Player,Share,predictions,Rk,Predicted_Rk
641,Nikola Jokić,0.961,0.154307,1,3
8624,Joel Embiid,0.580,0.162713,2,2
3651,Stephen Curry,0.449,0.142386,3,6
9907,Giannis Antetokounmpo,0.345,0.207436,4,1
1389,Chris Paul,0.138,0.072294,5,33
10997,Luka Dončić,0.042,0.151430,6,4
7464,Damian Lillard,0.038,0.116303,7,12
3536,Julius Randle,0.020,0.088878,8,24
3531,Derrick Rose,0.010,0.033000,9,76
11358,Rudy Gobert,0.008,0.095349,10,19


# Error Metric

In [98]:
def find_average_precision(combination):
    # Take the top 5 MVP winners
    actual = combination.sort_values("Share", ascending=False).head(5)
    predicted = combination.sort_values("predictions", ascending = False)
    ps = []
    found = 0
    seen = 1
    # if predicted player is in top 5, we get 100%, but if not, then penalize based on how far off
    # biased towards top of the ranking (rank in top 5 a lot more important than rank outside)
    for index, row in predicted.iterrows():
        if row["Player"] in actual["Player"].values:
            found += 1
            ps.append(found/seen)
        seen += 1
    return sum(ps) / len(ps)

In [99]:
# compares my top 5 to the actual top 5 i.e., chris paul is ranked 5 but we picked him at 33, so we get 0.15151515 for that
find_average_precision(combination)

0.7636363636363636

In [123]:
years = list(range(1991, 2022))

# Backtesting

In [124]:
# Backtesting is a way of testing if a model's predictions are in line with realised data - if historical is very different from predicted, model is not good
aps = []
all_predictions = []

# start on 5th year since we need some data to make predictions with
# train on all years immediately before current year, test on current year for each iteration on for loop
for year in years[5:]:
    train = stats[stats["Year"] < year]
    test = stats[stats["Year"] == year]
    reg.fit(train[predictors], train["Share"])
    predictions = reg.predict(test[predictors])
    # make predictions into a dataframe instead of just a bunch of numbers
    predictions = pd.DataFrame(predictions, columns = ['predictions'], index = test.index)
    # joins tables (columns) length wise, i.e., || -> |||||
    combination = pd.concat([test[["Player", "Share"]], predictions], axis = 1)
    all_predictions.append(combination)
    aps.append(find_average_precision(combination))

In [126]:
# mean average precision
sum(aps) / len(aps)

0.7112884360789578

In [131]:
# helpful for diagnosing if there is a common issue with our model - with what features it is thrown off the most by (ex. position)
def add_ranks(combination):
    combination = combination.sort_values("Share", ascending=False)
    combination["Rk"] = list(range(1, combination.shape[0]+1))
    combination = combination.sort_values("predictions", ascending = False)
    combination["Predicted_Rk"] = list(range(1, combination.shape[0] + 1))
    combination["Diff"] = combination["Rk"] - combination["Predicted_Rk"]
    return combination

In [137]:
ranking = add_ranks(all_predictions[1])
ranking[ranking["Rk"] <= 5].sort_values("Diff", ascending = False)

,Player,Share,predictions,Rk,Predicted_Rk,Diff
1600,Karl Malone,0.857,0.192318,1,2,-1
10524,Michael Jordan,0.832,0.167629,2,3,-1
908,Grant Hill,0.327,0.128646,3,6,-3
4682,Tim Hardaway,0.207,0.059984,4,20,-16
8248,Glen Rice,0.117,0.033110,5,53,-48


In [142]:
def backtest(stats, model, year, predictors):
    # Backtesting is a way of testing if a model's predictions are in line with realised data - if historical is very different from predicted, model is not good
    aps = []
    all_predictions = []

    # start on 5th year since we need some data to make predictions with
    # train on all years immediately before current year, test on current year for each iteration on for loop
    for year in years[5:]:
        train = stats[stats["Year"] < year]
        test = stats[stats["Year"] == year]
        model.fit(train[predictors], train["Share"])
        predictions = model.predict(test[predictors])
        # make predictions into a dataframe instead of just a bunch of numbers
        predictions = pd.DataFrame(predictions, columns = ['predictions'], index = test.index)
        # joins tables (columns) length wise, i.e., || -> |||||
        combination = pd.concat([test[["Player", "Share"]], predictions], axis = 1)
        combination = add_ranks(combination)
        all_predictions.append(combination)
        aps.append(find_average_precision(combination))
    return sum(aps)/len(aps), aps, pd.concat(all_predictions)

In [143]:
mean_ap, aps, all_predictions = backtest(stats, reg, years[5:], predictors)

In [144]:
mean_ap

0.7112884360789578

In [148]:
# If we have time - where does Jason Kidd differ from other candidates in terms of stats, why would he be placed like this?
all_predictions[all_predictions["Rk"] <= 5].sort_values("Diff").head(10)

,Player,Share,predictions,Rk,Predicted_Rk,Diff
1224,Jason Kidd,0.712,0.028209,2,52,-50
8248,Glen Rice,0.117,0.033110,5,53,-48
5175,Steve Nash,0.839,0.034100,1,45,-44
8516,Peja Stojaković,0.228,0.036269,4,38,-34
5193,Steve Nash,0.739,0.054128,1,34,-33
12726,Joakim Noah,0.258,0.046968,4,37,-33
3657,Chauncey Billups,0.344,0.052698,5,35,-30
1389,Chris Paul,0.138,0.072294,5,33,-28
5208,Steve Nash,0.785,0.074421,2,21,-19
4682,Tim Hardaway,0.207,0.059984,4,20,-16


In [149]:
# highest weighted features in the regression
reg.coef_

array([ 3.21225799e-04,  9.16295119e-05, -5.62287893e-06, -4.14969303e-03,
        5.69435368e-03,  5.51627207e-03, -1.35025345e-01,  4.56936500e-03,
       -1.03942800e-02, -9.82179223e-03,  1.69452132e-02, -1.73181709e-02,
        4.08318282e-03,  7.00021750e-02, -6.58501469e-03,  1.13509610e-02,
       -4.57492793e-03,  2.16104448e-02,  3.50411277e-02, -2.75059354e-02,
        7.45553644e-03,  1.16354220e-02,  1.12337946e-02, -9.69644454e-03,
       -2.63402385e-03,  5.89333750e-03, -1.84247808e-04,  9.78858335e-05,
       -2.95483880e-04,  2.71257474e-02,  2.69641841e-04, -5.56160227e-04,
       -2.18797486e-04, -5.21394525e-04])

In [152]:
pd.concat([pd.Series(reg.coef_), pd.Series(predictors)], axis = 1).sort_values(0, ascending = False)

,0,1
13,0.070002,eFG%
18,0.035041,DRB
29,0.027126,W/L%
17,0.021610,ORB
10,0.016945,2P
21,0.011635,STL
15,0.011351,FTA
22,0.011234,BLK
20,0.007456,AST
25,0.005893,PTS


In [153]:
# how much above the mean are we?
stat_ratios = stats[["PTS", "AST", "STL", "BLK", "3P", "Year"]].groupby("Year").apply(lambda x: x/x.mean())

In [154]:
stat_ratios

,PTS,AST,STL,BLK,3P,Year
0,1.013334,0.420714,0.961127,0.673469,0.508587,1.0
1,1.614653,1.028412,1.647646,0.673469,4.577279,1.0
2,0.311795,0.093492,0.274608,1.571429,0.000000,1.0
3,0.200440,0.186984,0.274608,0.000000,0.000000,1.0
4,2.383005,1.636110,1.784950,0.897959,1.525760,1.0
...,...,...,...,...,...,...
14087,0.735752,0.819562,0.479763,1.528302,0.650951,1.0
14088,0.071202,0.000000,0.000000,0.000000,0.130190,1.0
14089,1.281633,0.601012,1.119447,2.547170,0.520761,1.0
14090,0.474679,0.218550,0.319842,1.273585,0.650951,1.0


In [156]:
stats[["PTS_T", "AST_R", "STL_R", "BLK_R", "3P_R"]] = stat_ratios[["PTS", "AST", "STL", "BLK", "3P"]]

In [157]:
stats.head()

,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,FG%,...,W/L%,GB,PS/G,PA/G,SRS,PTS_T,AST_R,STL_R,BLK_R,3P_R
0,A.C. Green,PF,27,LAL,82,21,26.4,3.1,6.6,0.476,...,0.707,5.0,106.3,99.6,6.73,1.013334,0.420714,0.961127,0.673469,0.508587
1,Byron Scott,SG,29,LAL,82,82,32.1,6.1,12.8,0.477,...,0.707,5.0,106.3,99.6,6.73,1.614653,1.028412,1.647646,0.673469,4.577279
2,Elden Campbell,PF,22,LAL,52,0,7.3,1.1,2.4,0.455,...,0.707,5.0,106.3,99.6,6.73,0.311795,0.093492,0.274608,1.571429,0.000000
3,Irving Thomas,PF,25,LAL,26,0,4.2,0.7,1.9,0.340,...,0.707,5.0,106.3,99.6,6.73,0.200440,0.186984,0.274608,0.000000,0.000000
4,James Worthy,SF,29,LAL,78,74,38.6,9.2,18.7,0.492,...,0.707,5.0,106.3,99.6,6.73,2.383005,1.636110,1.784950,0.897959,1.525760


In [158]:
predictors += ["PTS_T", "AST_R", "STL_R", "BLK_R", "3P_R"]

In [159]:
mean_ap, aps, all_predictions = backtest(stats, reg, years[5:], predictors)

In [160]:
mean_ap

0.7208380973034985

In [163]:
stats["NPos"] = stats["Pos"].astype("category").cat.codes

In [164]:
stats["NTm"] = stats["Tm"].astype("category").cat.codes

In [162]:
stats["Pos"].unique()

array(['PF', 'SG', 'SF', 'PG', 'C', 'PG-SG', 'PF-SF', 'SG-PG', 'PF-C',
       'SG-SF', 'SF-PF', 'SF-SG', 'C-PF', 'SG-PF', 'PG-SF', 'SF-C'],
      dtype=object)

In [ ]:
stats["NTm"]

In [165]:
# Linear regression can't find relationship between categorical numbers, even when they are made to be 1, 2,3, etc.
# Need to use something else - in this case, we'll use random forest

In [166]:
# Random forest makes a series of decision trees and averages predictions from those trees

In [168]:
from sklearn.ensemble import RandomForestRegressor

#n_estimators is number of trees, random state = 1 means repeat the same outcome twice, min_samples_split means min number of values at a node to be split
rf = RandomForestRegressor(n_estimators = 100, random_state = 1, min_samples_split = 5)

mean_ap, aps, all_predictions = backtest(stats, rf, years[28:], predictors)

In [169]:
mean_ap

0.7239398208628978

In [170]:
mean_ap, aps, all_predictions = backtest(stats, reg, years[28:], predictors)

In [171]:
mean_ap

0.7208380973034985

In [180]:
train = stats[stats["Year"] < 2021]
test = stats[stats["Year"] == 2021]

In [181]:
rf.fit(train[predictors], train["Share"])
predictions = rf.predict(test[predictors])

In [182]:
predictions = pd.DataFrame(predictions, columns = ['predictions'], index = test.index)

In [183]:
combination = pd.concat([test[["Player", "Share"]], predictions], axis = 1)

In [184]:
combination.sort_values("Share", ascending = False).head(10)

,Player,Share,predictions
641,Nikola Jokić,0.961,0.209054
8624,Joel Embiid,0.580,0.274154
3651,Stephen Curry,0.449,0.252877
9907,Giannis Antetokounmpo,0.345,0.305397
1389,Chris Paul,0.138,0.007873
10997,Luka Dončić,0.042,0.142365
7464,Damian Lillard,0.038,0.103845
3536,Julius Randle,0.020,0.024870
3531,Derrick Rose,0.010,0.000000
11358,Rudy Gobert,0.008,0.016449
